In [ ]:
import pandas as pd
#part 1
df = pd.read_csv('sales_data.csv')
print(df.head())
print(df.tail())
print(df.columns)
print(df.dtypes)
numerical_cols = df.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = df.select_dtypes(exclude=["number"]).columns.tolist()
print("Numerical Columns:", numerical_cols)
print("Categorical Columns:", categorical_cols)
summary = df.describe()
print(summary)


In [ ]:
#part 2
print("missing values prior to cleaning: ")
print(df.isnull().sum())
print("total missing values:",df.isnull().sum().sum())
for col in numerical_cols:
    df[col] = df[col].fillna(df[col].median())
for col in categorical_cols:
    if df[col].isnull().any() and not df[col].mode().empty:
        df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values after cleaning:")
print(df.isnull().sum())
print("Duplicate records before removal:", df.duplicated().sum())

df = df.drop_duplicates()

print("Duplicate records after removal:", df.duplicated().sum())

print("Data types before correction:")
print(df.dtypes)

if 'Order Date' in df.columns:
    df['Order Date'] = pd.to_datetime(
        df['Order Date'], errors='coerce'
    )

print("Data types after correction:")
print(df.dtypes)
for col in numerical_cols:
    print(f"Negative values in {col}:")
    print((df[col] < 0).sum())

for col in categorical_cols:
    print(f"Unique values in {col}:")
    print(df[col].value_counts(dropna=False).head(20))

for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].str.strip()

#Verifying the dataset after cleaning
print("First 5 rows:")
print(df.head())
print("Dataset shape (rows, columns):")
print(df.shape)
print("Missing values:")
print(df.isnull().sum())
print("Total missing values:", df.isnull().sum().sum())
print("Duplicate records:", df.duplicated().sum())
print("Final data types:")
print(df.dtypes)
print("Final summary statistics:")
print(df.describe(include="all"))
df.to_csv('sales_data_cleaned.csv', index=False)
print("Cleaned dataset saved as sales_data_cleaned.csv")

In [ ]:
#part 3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('sales_data_cleaned.csv')

numerical_cols = df.select_dtypes(include='number').columns.tolist()
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

print("Numerical Features")
print(df[numerical_cols].describe().T)

for col in numerical_cols:
    print(col, "Minimum:", df[col].min())
    print(col, "Maximum:", df[col].max())
    print(col, "Average:", df[col].mean())
    print(col, "Median:", df[col].median())

print("Categorical Features")

for col in categorical_cols:
    print(col, "Unique Categories:", df[col].nunique())
    print(df[col].value_counts().head(10))

for col in numerical_cols:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[col].dropna(), kde=True)
    plt.title(f"Distribution of {col}")
    plt.tight_layout()
    plt.show()

for col in categorical_cols:
    if df[col].nunique() <= 20:
        plt.figure(figsize=(8, 4))
        df[col].value_counts().plot(kind='bar')
        plt.title(f"Distribution of {col}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

print("Outlier Analysis")

for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(col, "Outliers:", len(outliers))
    print("Lower Bound:", lower_bound)
    print("Upper Bound:", upper_bound)

if len(numerical_cols) > 1:
    plt.figure(figsize=(10, 6))
    sns.heatmap(df[numerical_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
    plt.title("Correlation Between Numerical Features")
    plt.tight_layout()
    plt.show()


sales_cols = [
    col for col in df.columns
    if any(word in col.lower() for word in ['sales', 'revenue', 'profit', 'quantity'])
    and pd.api.types.is_numeric_dtype(df[col])
]

print("Important Business Features:", sales_cols)

for target in sales_cols:
    for category in categorical_cols:
        if df[category].nunique() <= 30:
            print(f"Average {target} by {category}")
            result = df.groupby(category)[target].mean()
            print(result.sort_values(ascending=False).head(10))


df[numerical_cols].describe().T.to_csv('numerical_feature_summary.csv')


In [ ]:
#part 4
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('sales_data_cleaned.csv')

numerical_cols = df.select_dtypes(include='number').columns.tolist()
corr = df[numerical_cols].corr()

print("Correlation Matrix:")
print(corr.round(2))

plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

strong_positive = []
strong_negative = []
weak_relationships = []

for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):
        col1 = corr.columns[i]
        col2 = corr.columns[j]
        value = corr.iloc[i, j]

        if pd.isna(value):
            continue

        if value >= 0.7:
            strong_positive.append((col1, col2, round(value, 2)))
        elif value <= -0.7:
            strong_negative.append((col1, col2, round(value, 2)))
        elif abs(value) < 0.3:
            weak_relationships.append((col1, col2, round(value, 2)))

print("\nStrong Positive Relationships:")
print(strong_positive if strong_positive else "None found")

print("\nStrong Negative Relationships:")
print(strong_negative if strong_negative else "None found")

print("\nWeak Relationships:")
print(weak_relationships if weak_relationships else "None found")

print("\nImportant Feature Relationships:")
for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):
        value = corr.iloc[i, j]
        if pd.notna(value) and abs(value) >= 0.5:
            print(f"{corr.columns[i]} and {corr.columns[j]}: {value:.2f}")

In [ ]:
#part 5
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('sales_data_cleaned.csv')

sns.set_theme(style="whitegrid")
df.columns = df.columns.str.strip()

numerical_cols = df.select_dtypes(include='number').columns.tolist()
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

date_cols = []
for col in df.columns:
    if any(word in col.lower() for word in ['date', 'time']):
        converted = pd.to_datetime(df[col], errors='coerce')
        if converted.notna().sum() > 0:
            df[col] = converted
            date_cols.append(col)

categorical_cols = [col for col in categorical_cols if col not in date_cols]

sales_cols = [
    col for col in numerical_cols
    if any(word in col.lower() for word in ['sales', 'revenue', 'profit', 'quantity', 'amount'])
]

value_col = sales_cols[0] if sales_cols else (
    numerical_cols[0] if numerical_cols else None
)

category_col = next(
    (col for col in categorical_cols
     if any(word in col.lower() for word in ['category', 'product', 'region', 'segment', 'city'])),
    categorical_cols[0] if categorical_cols else None
)

def finish_plot():
    plt.tight_layout()
    plt.show()

if value_col:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[value_col].dropna(), bins=25, kde=True)
    plt.title(f'1. Distribution of {value_col}')
    plt.xlabel(value_col)
    plt.ylabel('Frequency')
    finish_plot()

if category_col and value_col:
    top = df.groupby(category_col, observed=True)[value_col].sum().nlargest(10)
    plt.figure(figsize=(9, 5))
    sns.barplot(x=top.values, y=top.index, color='steelblue')
    plt.title(f'2. Top Categories by Total {value_col}')
    plt.xlabel(f'Total {value_col}')
    plt.ylabel(category_col)
    finish_plot()

if category_col:
    counts = df[category_col].value_counts().head(15)
    plt.figure(figsize=(10, 5))
    sns.countplot(data=df[df[category_col].isin(counts.index)],
                  y=category_col, order=counts.index, color='cornflowerblue')
    plt.title(f'3. Record Count by {category_col}')
    plt.xlabel('Number of Records')
    plt.ylabel(category_col)
    finish_plot()

if category_col:
    pie_data = df[category_col].value_counts().head(6)
    if len(df[category_col].dropna().unique()) > 6:
        pie_data.loc['Other'] = len(df[category_col].dropna()) - pie_data.sum()
    plt.figure(figsize=(7, 7))
    plt.pie(pie_data, labels=pie_data.index, autopct='%1.1f%%',
            startangle=90)
    plt.title(f'4. Share of Records by {category_col}')
    plt.ylabel('')
    finish_plot()

if len(numerical_cols) >= 2:
    x_col = next((c for c in numerical_cols if c.lower() in ['quantity', 'units', 'discount']), numerical_cols[0])
    y_col = next((c for c in sales_cols if c != x_col), numerical_cols[1])
    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=df, x=x_col, y=y_col, alpha=0.65)
    plt.title(f'5. Relationship Between {x_col} and {y_col}')
    plt.xlabel(x_col)
    plt.ylabel(y_col)
    finish_plot()

if date_cols and value_col:
    date_col = date_cols[0]
    trend = df.dropna(subset=[date_col]).copy()
    trend = trend.set_index(date_col).resample('ME')[value_col].sum()
    if not trend.empty:
        plt.figure(figsize=(10, 5))
        plt.plot(trend.index, trend.values, marker='o')
        plt.title(f'6. Monthly {value_col} Trend')
        plt.xlabel('Month')
        plt.ylabel(f'Total {value_col}')
        plt.xticks(rotation=45)
        finish_plot()

if category_col and value_col:
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=df, x=category_col, y=value_col)
    plt.title(f'7. Distribution of {value_col} Across {category_col}')
    plt.xlabel(category_col)
    plt.ylabel(value_col)
    plt.xticks(rotation=45)
    finish_plot()

if category_col and value_col:
    plt.figure(figsize=(10, 5))
    sns.violinplot(data=df, x=category_col, y=value_col, cut=0)
    plt.title(f'8. Density of {value_col} Across {category_col}')
    plt.xlabel(category_col)
    plt.ylabel(value_col)
    plt.xticks(rotation=45)
    finish_plot()

if len(numerical_cols) >= 2:
    plt.figure(figsize=(10, 7))
    sns.heatmap(df[numerical_cols].corr(), annot=True, cmap='coolwarm',
                fmt='.2f', vmin=-1, vmax=1)
    plt.title('9. Numerical Feature Correlation Heatmap')
    finish_plot()

if len(numerical_cols) >= 2:
    pair_cols = numerical_cols[:4]
    pair_df = df[pair_cols].dropna()
    if len(pair_df) > 1:
        sns.pairplot(pair_df, diag_kind='hist', corner=True)
        plt.suptitle('10. Relationships Between Numerical Features', y=1.02)
        plt.show()

if value_col:
    plt.figure(figsize=(8, 5))
    sns.kdeplot(data=df, x=value_col, fill=True)
    plt.title(f'11. Density Distribution of {value_col}')
    plt.xlabel(value_col)
    plt.ylabel('Density')
    finish_plot()

if date_cols and value_col:
    date_col = date_cols[0]
    trend_df = df.dropna(subset=[date_col]).copy()
    trend_df = trend_df.set_index(date_col).resample('ME')[value_col].sum()
    if not trend_df.empty:
        plt.figure(figsize=(10, 5))
        plt.fill_between(trend_df.index, trend_df.values, alpha=0.4, color='teal')
        plt.plot(trend_df.index, trend_df.values, color='teal')
        plt.title(f'12. Monthly {value_col} Area Chart')
        plt.xlabel('Month')
        plt.ylabel(f'Total {value_col}')
        plt.xticks(rotation=45)
        finish_plot()


In [ ]:

import pandas as pd
import numpy as np

df = pd.read_csv('sales_data_cleaned.csv')
df.columns = df.columns.str.strip()

num_cols = df.select_dtypes(include='number').columns.tolist()
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

sales_col = next((c for c in num_cols if 'sales' in c.lower() or 'revenue' in c.lower()), None)
profit_col = next((c for c in num_cols if 'profit' in c.lower()), None)
qty_col = next((c for c in num_cols if 'quantity' in c.lower() or 'units' in c.lower()), None)
discount_col = next((c for c in num_cols if 'discount' in c.lower()), None)

print("1. Dataset overview")
print("Rows and columns:", df.shape)
print("Date range and missing values:")
print(df.isnull().sum())

print("\n2. Numerical summary")
print(df[num_cols].describe().round(2))

print("\n3. Category performance")
for col in cat_cols:
    if df[col].nunique() <= 100:
        print(f"\nCategory: {col}")
        if sales_col:
            print("Highest sales:")
            print(df.groupby(col)[sales_col].sum().sort_values(ascending=False).head(5))
            print("Lowest sales:")
            print(df.groupby(col)[sales_col].sum().sort_values().head(5))
        if profit_col:
            print("Highest profit:")
            print(df.groupby(col)[profit_col].sum().sort_values(ascending=False).head(5))
            print("Lowest profit:")
            print(df.groupby(col)[profit_col].sum().sort_values().head(5))

print("\n4. Sales and profit indicators")
if sales_col:
    print("Total sales:", round(df[sales_col].sum(), 2))
    print("Average sales:", round(df[sales_col].mean(), 2))
    print("Median sales:", round(df[sales_col].median(), 2))
    print("Highest transaction:", round(df[sales_col].max(), 2))
    print("Lowest transaction:", round(df[sales_col].min(), 2))

if profit_col:
    print("Total profit:", round(df[profit_col].sum(), 2))
    print("Average profit:", round(df[profit_col].mean(), 2))
    if sales_col:
        total_sales = df[sales_col].sum()
        print("Overall profit margin (%):",
              round(df[profit_col].sum() / total_sales * 100, 2)
              if total_sales != 0 else "Undefined")

if qty_col:
    print("Total quantity:", df[qty_col].sum())
    print("Average quantity:", round(df[qty_col].mean(), 2))

if discount_col:
    print("Average discount:", round(df[discount_col].mean(), 2))
    if profit_col:
        print("Discount-profit correlation:",
              round(df[discount_col].corr(df[profit_col]), 3))

print("\n5. Correlation analysis")
if len(num_cols) > 1:
    corr = df[num_cols].corr()
    pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack()
    print(pairs.reindex(pairs.abs().sort_values(ascending=False).index).head(15).round(3))

print("\n6. Outlier analysis")
for col in num_cols:
    s = df[col].dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    count = ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()
    print(col, "Potential outliers:", int(count))

print("\n7. Date trends")
date_cols = []
for col in df.columns:
    if 'date' in col.lower():
        dates = pd.to_datetime(df[col], errors='coerce')
        if dates.notna().sum() > 0:
            df[col] = dates
            date_cols.append(col)

if date_cols and sales_col:
    date_col = date_cols[0]
    monthly = df.dropna(subset=[date_col]).set_index(date_col)[sales_col].resample('ME').sum()
    print("Monthly sales:")
    print(monthly)
    print("Best month:", monthly.idxmax(), monthly.max()) if not monthly.empty else None
    print("Lowest month:", monthly.idxmin(), monthly.min()) if not monthly.empty else None

print("\n8. Customer analysis")
customer_col = next((c for c in cat_cols if 'customer' in c.lower()), None)
if customer_col:
    print("Unique customers:", df[customer_col].nunique())
    if sales_col:
        print("Top customers by sales:")
        print(df.groupby(customer_col)[sales_col].sum().sort_values(ascending=False).head(10))
    if profit_col:
        print("Top customers by profit:")
        print(df.groupby(customer_col)[profit_col].sum().sort_values(ascending=False).head(10))
